# CS781 Phase 6 · Model 4 · LoRMA

**Complete conditional-generation experiment notebook** based on the user's original Phase-1 Model 4 architecture and BPE tokenizer.

**Your design decisions to review:** adaptation method and target layers, prompt format, data selection, source/summary budgets, optimization hyperparameters, distillation teacher and output decoding. Values supplied below are editable starting settings, **not optimized values**. No pretrained external language model is loaded.

**Method reference:** Bihany, Patel, Modi (ACL 2025), LoRMA: https://aclanthology.org/2025.findings-acl.527/

**Execution:** enable a Kaggle GPU and internet, attach matching Phase-1 assets containing `pytorch_model.bin` and `custom_bpe_scratch.json`, and add Hugging Face token to Kaggle Secret `HF`.

*If running on Colab*, change `CFG["asset_dir"]` and `CFG["output_dir"]` to your mounted paths. The official repo is `Exploration-Lab/CS-781-Capstone`.

**Note:** Full-size GPU training and actual gated dataset access cannot be verified in this offline environment. Do not interpret syntax/synthetic checks as real performance.

In [17]:
# 1 · Experiment configuration (review these choices yourself)
MODEL_ID = 4
METHOD = "lorma"

CFG = {
    "asset_dir": "/kaggle/input/datasets/kalyanbabuk/phase1m4",
    "output_dir": "/kaggle/working/phase6_model4_lorma",
    "teacher_checkpoint": "/kaggle/input/phase6-model4-lora/adapter_best.pt",
    "train_teacher_if_missing": True,
    "vocab_size": 32000,
    "hidden_size": 560,
    "n_layers": 15,
    "n_heads": 8,
    "n_kv_heads": 4,
    "model_context": 2048,
    "student_layers": 8,
    "max_source_tokens": 640,
    "max_target_tokens": 192,
    "max_new_tokens": 180,
    "batch_size": 1,
    "gradient_accumulation": 4,
    "epochs": 3,
    "student_epochs": 3,
    "learning_rate": 2e-4,
    "student_learning_rate": 2e-4,
    "weight_decay": 0.01,
    "rank": 8,
    "alpha": 16.0,
    "adapter_dropout": 0.05,
    "adapter_bottleneck": 64,
    "kd_temperature": 2.0,
    "kd_hard_weight": 0.6,
    "grad_clip": 1.0,
    "max_validation_batches": 40,
    "validation_generation_count": 30,
    "log_every": 50,
    "seed": 781,
}

# If the official dataset is already mounted, set this to its root directory.
# Otherwise, download from Hugging Face using Kaggle Secret HF.
LOCAL_DATA_ROOT = None

from pathlib import Path
OUTPUT_DIR = Path(CFG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_ID)
print("Method:", METHOD)
print("Outputs:", OUTPUT_DIR)


Model: 4
Method: lorma
Outputs: /kaggle/working/phase6_model4_lorma


In [18]:
CFG.update({
    "max_source_tokens": 768,
    "max_target_tokens": 1152,
    "max_new_tokens": 1152,
    "model_context": 2048,
    "batch_size": 1,
    "output_dir": "/kaggle/working/phase6_model2_lora_v2",
})

assert (
    CFG["max_source_tokens"]
    + CFG["max_target_tokens"]
    + 40
    <= CFG["model_context"]
)

## 2 · Original custom Phase-1 byte-level BPE tokenizer

In [19]:
"""Original Phase-1-compatible byte-level BPE inference tokenizer.

Vocabulary and merge ordering match the student's Phase-1 saved tokenizer.
This file does NOT train a new tokenizer.
"""

import json
import re
from pathlib import Path


class CustomByteLevelBPE:
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.inverse_vocab = {v: k for k, v in self.vocab.items()}
        self.next_id = 256
        self.pad_id = 0

    def load(self, path):
        with Path(path).open(encoding="utf-8") as stream:
            payload = json.load(stream)

        self.vocab_size = int(payload["vocab_size"])
        self.next_id = int(payload["next_id"])
        self.merges = {
            tuple(map(int, pair.split(","))): int(rank)
            for pair, rank in payload["merges"].items()
        }
        self.vocab = {
            int(index): bytes(data)
            for index, data in payload["vocab"].items()
        }
        self.inverse_vocab = {token: index for index, token in self.vocab.items()}
        return self

    def encode(self, text):
        encoded = []
        for chunk in re.findall(r"\S+|\s+", text):
            ids = list(chunk.encode("utf-8"))

            while len(ids) >= 2:
                pairs = zip(ids, ids[1:])
                possible = [
                    (self.merges[pair], pair)
                    for pair in pairs
                    if pair in self.merges
                ]
                if not possible:
                    break

                _, best_pair = min(possible)
                new_ids = []
                position = 0
                while position < len(ids):
                    if (
                        position + 1 < len(ids)
                        and (ids[position], ids[position + 1]) == best_pair
                    ):
                        new_ids.append(self.merges[best_pair])
                        position += 2
                    else:
                        new_ids.append(ids[position])
                        position += 1
                ids = new_ids

            encoded.extend(ids)
        return encoded

    def decode(self, ids):
        raw = b"".join(
            self.vocab[int(index)]
            for index in ids
            if int(index) != self.pad_id
        )
        return raw.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return len(self.vocab)


## 3 · Phase-1 checkpoint-compatible backbone

Model 2 and Model 4 preserve the exact trainable parameter names and module dimensions from the earlier notebooks. Sink/window causal masking also handles incremental generation correctly.

In [20]:
"""Checkpoint-compatible Model 2 and Model 4 Phase-1 architectures.

Model 2: RoPE + RMSNorm + QK Norm + GQA + SiLU FFN.
Model 4: RoPE + RMSNorm + GQA + attention-sink/local window + alternating FFNs.
All parameter/buffer names preserve the existing Phase-3 Model2/4 notebooks.
"""

import math

import torch
from torch import nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.square().mean(-1, keepdim=True) + self.eps) * self.weight


def precompute_rope(dim, max_len, theta=10000.0):
    frequencies = 1.0 / theta ** (
        torch.arange(0, dim, 2).float() / dim
    )
    positions = torch.arange(max_len)
    angles = torch.outer(positions, frequencies)
    return torch.view_as_real(torch.polar(torch.ones_like(angles), angles))


def apply_rope(query, key, freqs):
    q_complex = torch.view_as_complex(query.float().reshape(*query.shape[:-1], -1, 2))
    k_complex = torch.view_as_complex(key.float().reshape(*key.shape[:-1], -1, 2))
    freqs = freqs.view(1, query.shape[1], 1, -1)
    q_rot = torch.view_as_real(q_complex * freqs).flatten(-2)
    k_rot = torch.view_as_real(k_complex * freqs).flatten(-2)
    return q_rot.to(query.dtype), k_rot.to(key.dtype)


class ManualTransformerAttention(nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        self.scale = 1.0 / math.sqrt(head_dim)

    def forward(self, q, k, v, is_causal=True, attention_mask=None):
        query_length = q.size(-2)
        key_length = k.size(-2)
        scores = (q @ k.transpose(-2, -1)) * self.scale

        if is_causal and query_length > 1:
            query_positions = torch.arange(
                key_length - query_length, key_length, device=q.device
            )
            key_positions = torch.arange(key_length, device=q.device)
            causal = key_positions[None, :] <= query_positions[:, None]
            scores = scores.masked_fill(~causal, float("-inf"))

        if attention_mask is not None:
            mask = attention_mask[:, None, None, :].bool()
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(scores.float(), dim=-1).to(scores.dtype)
        return weights @ v


class GroupedQueryAttention(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.wq = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)
        self.q_norm = RMSNorm(n_heads * self.head_dim)
        self.k_norm = RMSNorm(n_kv_heads * self.head_dim)
        self.manual_attention = ManualTransformerAttention(self.head_dim)

    def forward(self, x, freqs_cis, kv_cache=None, attention_mask=None):
        batch, length, _ = x.shape
        q = self.q_norm(self.wq(x)).view(batch, length, self.n_heads, self.head_dim)
        k = self.k_norm(self.wk(x)).view(batch, length, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(batch, length, self.n_kv_heads, self.head_dim)
        q, k = apply_rope(q, k, freqs_cis)

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=1)
            v = torch.cat([kv_cache[1], v], dim=1)

        repeated_k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=2)
        repeated_v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=2)
        out = self.manual_attention(
            q.transpose(1, 2),
            repeated_k.transpose(1, 2),
            repeated_v.transpose(1, 2),
            is_causal=True,
            attention_mask=attention_mask,
        )
        out = out.transpose(1, 2).contiguous().view(batch, length, -1)
        return self.wo(out), (k, v)


class CustomTransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.attn = GroupedQueryAttention(dim, n_heads, n_kv_heads)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4, bias=False),
            nn.SiLU(),
            nn.Linear(dim * 4, dim, bias=False),
        )

    def forward(self, x, freqs_cis, kv_cache=None, attention_mask=None):
        attended, cache = self.attn(
            self.norm1(x), freqs_cis, kv_cache, attention_mask
        )
        x = x + attended
        return x + self.ffn(self.norm2(x)), cache


class SinkWindowAttention(nn.Module):
    def __init__(self, head_dim, window_size=256, sink_size=4):
        super().__init__()
        self.scale = 1.0 / math.sqrt(head_dim)
        self.window_size = window_size
        self.sink_size = sink_size

    def forward(self, q, k, v, is_causal=True, attention_mask=None):
        q_len, k_len = q.size(-2), k.size(-2)
        scores = (q @ k.transpose(-2, -1)) * self.scale

        if is_causal:
            query_positions = torch.arange(
                k_len - q_len, k_len, device=q.device
            )[:, None]
            key_positions = torch.arange(k_len, device=q.device)[None, :]
            causal = key_positions <= query_positions
            local = query_positions - key_positions < self.window_size
            sink = key_positions < self.sink_size
            visible = causal & (local | sink)
            scores = scores.masked_fill(~visible, float("-inf"))

        if attention_mask is not None:
            scores = scores.masked_fill(
                ~attention_mask[:, None, None, :].bool(), float("-inf")
            )

        weights = F.softmax(scores.float(), dim=-1).to(scores.dtype)
        return weights @ v


class EfficientGQA(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads, window_size=256, sink_size=4):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.wq = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)
        self.attention = SinkWindowAttention(
            self.head_dim, window_size, sink_size
        )

    def forward(self, x, freqs_cis, kv_cache=None, attention_mask=None):
        batch, length, _ = x.shape
        q = self.wq(x).view(batch, length, self.n_heads, self.head_dim)
        k = self.wk(x).view(batch, length, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(batch, length, self.n_kv_heads, self.head_dim)
        q, k = apply_rope(q, k, freqs_cis)

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=1)
            v = torch.cat([kv_cache[1], v], dim=1)

        cache = (k, v)
        k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=2)
        v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=2)
        attended = self.attention(
            q.transpose(1, 2),
            k.transpose(1, 2),
            v.transpose(1, 2),
            is_causal=True,
            attention_mask=attention_mask,
        )
        out = attended.transpose(1, 2).contiguous().view(batch, length, -1)
        return self.wo(out), cache


class StandardFFN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 4, bias=False),
            nn.GELU(),
            nn.Linear(dim * 4, dim, bias=False),
        )

    def forward(self, x):
        return self.net(x)


class BottleneckFFN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim // 4, bias=False),
            nn.GELU(),
            nn.Linear(dim // 4, dim, bias=False),
        )

    def forward(self, x):
        return self.net(x)


class EfficientTransformerBlock(nn.Module):
    def __init__(
        self, dim, n_heads, n_kv_heads, window_size, sink_size, is_bottleneck=False
    ):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = EfficientGQA(
            dim, n_heads, n_kv_heads, window_size, sink_size
        )
        self.norm2 = RMSNorm(dim)
        self.ffn = BottleneckFFN(dim) if is_bottleneck else StandardFFN(dim)

    def forward(self, x, freqs_cis, kv_cache=None, attention_mask=None):
        attended, cache = self.attn(
            self.norm1(x), freqs_cis, kv_cache, attention_mask
        )
        x = x + attended
        return x + self.ffn(self.norm2(x)), cache


class CustomLegalModel(nn.Module):
    def __init__(
        self,
        model_id,
        vocab_size=32000,
        dim=None,
        n_layers=15,
        n_heads=8,
        n_kv_heads=4,
        max_seq_len=2048,
        window_size=256,
        sink_size=4,
    ):
        super().__init__()
        if model_id not in (2, 4):
            raise ValueError("model_id must be 2 or 4")
        self.model_id = model_id
        self.dim = dim if dim is not None else (512 if model_id == 2 else 560)
        self.max_seq_len = max_seq_len

        if model_id == 2:
            self.tok_embeddings = nn.Embedding(vocab_size, self.dim)
            self.layers = nn.ModuleList([
                CustomTransformerBlock(self.dim, n_heads, n_kv_heads)
                for _ in range(n_layers)
            ])
            tied_embedding = self.tok_embeddings
        else:
            self.embeddings = nn.Embedding(vocab_size, self.dim)
            self.layers = nn.ModuleList([
                EfficientTransformerBlock(
                    self.dim,
                    n_heads,
                    n_kv_heads,
                    window_size,
                    sink_size,
                    is_bottleneck=index % 2 != 0,
                )
                for index in range(n_layers)
            ])
            tied_embedding = self.embeddings

        self.norm = RMSNorm(self.dim)
        self.output = nn.Linear(self.dim, vocab_size, bias=False)
        self.output.weight = tied_embedding.weight
        self.register_buffer(
            "freqs_cis", precompute_rope(self.dim // n_heads, max_seq_len)
        )

    def forward(self, tokens, start_pos=0, kv_caches=None, attention_mask=None):
        batch, length = tokens.shape
        if start_pos + length > self.max_seq_len:
            raise ValueError("Sequence exceeds configured RoPE context length")

        freqs = torch.view_as_complex(
            self.freqs_cis[start_pos:start_pos + length].to(tokens.device)
        )
        embedding = (
            self.tok_embeddings if self.model_id == 2 else self.embeddings
        )
        states = embedding(tokens)
        new_caches = []

        for index, layer in enumerate(self.layers):
            previous_cache = None if kv_caches is None else kv_caches[index]
            states, cache = layer(
                states, freqs, previous_cache, attention_mask=attention_mask
            )
            new_caches.append(cache)

        return self.output(self.norm(states)), new_caches


## 4 · PEFT method implementations

The active technique is selected by `METHOD`. LoRMA uses a **multiplicative identity-inflated update**, not a renamed LoRA operation.

In [21]:
"""Three genuinely different PEFT methods for Phase-6 generation.

LoRA: additive low-rank update of a frozen linear layer.
LoRMA: identity-inflated low-rank multiplicative transform of W0.
Pfeiffer: bottleneck residual adapter after the transformer FFN.
"""

import math
import types

import torch
from torch import nn
import torch.nn.functional as F


class LoRALinear(nn.Module):
    def __init__(self, original, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = original
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)
        # Allocate new adapter weights where the pretrained projection lives.
        # After wrapping a CUDA linear layer, torch.empty() defaults to CPU.
        self.A = nn.Parameter(
            original.weight.new_empty((rank, original.in_features))
        )
        self.B = nn.Parameter(
            original.weight.new_zeros((original.out_features, rank))
        )
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        low_rank = F.linear(self.dropout(x), self.A)
        return self.base(x) + self.scale * F.linear(low_rank, self.B)


class LoRMALinear(nn.Module):
    """Identity-inflated LoRMA: W' = (I + alpha/r * B A) W0.

    The low-rank transformation multiplies the frozen original weight
    on the output side. This is distinct from additive LoRA's W0 + BA.
    """

    def __init__(self, original, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = original
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)
        # Match the device/dtype of the frozen model weight.
        self.A = nn.Parameter(
            original.weight.new_empty((rank, original.out_features))
        )
        self.B = nn.Parameter(
            original.weight.new_zeros((original.out_features, rank))
        )
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        base_output = self.base(x)
        rotated = F.linear(self.dropout(base_output), self.A)
        return base_output + self.scale * F.linear(rotated, self.B)


class PfeifferAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck=64, dropout=0.05):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.up = nn.Linear(bottleneck, hidden_size)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, hidden):
        return hidden + self.up(self.drop(self.act(self.down(hidden))))


def add_pfeiffer_to_block(block, *, hidden_size, bottleneck, dropout):
    # The adapter is created after the backbone may already be on CUDA.
    reference = next(block.parameters())
    block.pfeiffer_adapter = PfeifferAdapter(
        hidden_size=hidden_size,
        bottleneck=bottleneck,
        dropout=dropout,
    ).to(device=reference.device, dtype=reference.dtype)
    old_forward = block.forward

    def new_forward(self, x, freqs_cis, kv_cache=None, attention_mask=None):
        output, cache = old_forward(
            x, freqs_cis, kv_cache, attention_mask=attention_mask
        )
        return self.pfeiffer_adapter(output), cache

    block.forward = types.MethodType(new_forward, block)


def apply_adaptation(model, method, config):
    """Freezes Phase-1 weights and installs trainable method-specific components."""
    if method not in {"lora", "lorma", "pfeiffer"}:
        raise ValueError(f"Unknown PEFT technique: {method}")

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    if method in {"lora", "lorma"}:
        layer_type = LoRALinear if method == "lora" else LoRMALinear
        for block in model.layers:
            for name in ("wq", "wk", "wv", "wo"):
                original = getattr(block.attn, name)
                setattr(
                    block.attn,
                    name,
                    layer_type(
                        original,
                        rank=config["rank"],
                        alpha=config["alpha"],
                        dropout=config["adapter_dropout"],
                    ),
                )
    else:
        for block in model.layers:
            add_pfeiffer_to_block(
                block,
                hidden_size=model.dim,
                bottleneck=config["adapter_bottleneck"],
                dropout=config["adapter_dropout"],
            )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if not trainable:
        raise AssertionError("The adapter left zero trainable parameters")
    return model


def trainable_parameter_dict(model):
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }


def restore_trainable_parameters(model, saved):
    named = dict(model.named_parameters())
    expected = {key for key, value in named.items() if value.requires_grad}
    if set(saved) != expected:
        raise RuntimeError(
            f"Adapter parameter mismatch: missing={sorted(expected - set(saved))[:6]} "
            f"extra={sorted(set(saved) - expected)[:6]}"
        )

    with torch.no_grad():
        for key, value in saved.items():
            if tuple(named[key].shape) != tuple(value.shape):
                raise RuntimeError(f"Wrong adapter shape: {key}")
            named[key].copy_(value.to(named[key].device, dtype=named[key].dtype))


def merge_trained_linear(module):
    """Get the trained effective weight for KD initialization."""
    if isinstance(module, LoRALinear):
        return module.base.weight + module.scale * (module.B @ module.A)
    if isinstance(module, LoRMALinear):
        identity_inflation = module.B @ module.A
        return module.base.weight + module.scale * (
            identity_inflation @ module.base.weight
        )
    if isinstance(module, nn.Linear):
        return module.weight
    raise TypeError(f"Unsupported student initialization layer {type(module)}")


## 5 · Official data and conditional summarization trainer

The prompt tokens are masked out of supervised cross entropy. Only summary tokens and the end marker contribute to the objective. The KD student also learns from the teacher's softened next-token distributions.

In [22]:
"""Phase-6 conditional language-model data construction and official dataset I/O."""

from pathlib import Path
import json
import os
import random
import re

import torch
from torch.utils.data import Dataset


REPO_ID = "Exploration-Lab/CS-781-Capstone"
END_SUMMARY = "\n[END SUMMARY]"
PREFIX = (
    "Summarize the following Indian legal judgment accurately. "
    "Cover the facts, the central legal issue, the court reasoning, "
    "and the outcome. Avoid introducing facts that are not in the judgment.\n\n"
    "JUDGMENT:\n"
)
SUFFIX = "\n\nSUMMARY:\n"


def get_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        return None


def phase6_path(split, local_root=None):
    if split not in {"train", "test"}:
        raise ValueError("Only train and test are official Phase-6 files")

    relative = f"Phase_6_Summ/{split.title()}/{split}.json"
    if local_root is not None:
        base = Path(local_root)
        for candidate in (
            base / relative,
            base / split.title() / f"{split}.json",
            base / f"{split}.json",
        ):
            if candidate.is_file():
                return candidate

    from huggingface_hub import hf_hub_download
    return Path(
        hf_hub_download(
            repo_id=REPO_ID,
            filename=relative,
            repo_type="dataset",
            token=get_hf_token(),
        )
    )


def load_data(split, local_root=None):
    with phase6_path(split, local_root).open(encoding="utf-8") as stream:
        rows = json.load(stream)

    if isinstance(rows, dict) and "data" in rows:
        rows = rows["data"]
    if not isinstance(rows, list):
        raise TypeError("Expected official JSON list")

    seen = set()
    for record in rows:
        if not isinstance(record, dict) or not isinstance(record.get("id"), str):
            raise ValueError("Every example needs a string id")
        if not isinstance(record.get("document"), str):
            raise ValueError("Every example needs a document string")
        if split == "train" and not isinstance(record.get("summary"), str):
            raise ValueError("Every training example needs a summary string")
        if record["id"] in seen:
            raise ValueError(f"Repeated document id {record['id']}")
        seen.add(record["id"])
    return rows


def split_train_validation(rows, fraction=0.10, seed=781):
    if len(rows) < 2:
        raise ValueError("Need at least two training examples")
    indices = list(range(len(rows)))
    random.Random(seed).shuffle(indices)
    n_val = max(1, round(len(rows) * fraction))
    selected = set(indices[:min(n_val, len(rows) - 1)])
    training = [row for index, row in enumerate(rows) if index not in selected]
    validation = [row for index, row in enumerate(rows) if index in selected]
    return training, validation


def select_document_text(document):
    """Lead + judgment-tail + decision-related snippets for long cases.

    This selection is deterministic and uses *only* the given document.
    """
    text = re.sub(r"\s+", " ", document).strip()
    if len(text) <= 4400:
        return text

    lead = text[:2100]
    tail = text[-1300:]
    keywords = re.compile(
        r"\b(held|therefore|accordingly|in conclusion|appeal is|petition is|"
        r"we find|we hold|judgment|order|dismissed|allowed)\b",
        flags=re.IGNORECASE,
    )
    middle = text[2100:-1300]
    match = keywords.search(middle)
    evidence = "" if match is None else middle[max(0, match.start() - 280):match.end() + 480]
    return f"{lead}\n[SELECTED REASONING]\n{evidence}\n[FINAL SECTION]\n{tail}"


def make_prompt_ids(document, tokenizer, max_source_tokens):
    prefix_ids = tokenizer.encode(PREFIX)
    suffix_ids = tokenizer.encode(SUFFIX)

    budget = (
        max_source_tokens
        - len(prefix_ids)
        - len(suffix_ids)
    )

    if budget < 32:
        raise ValueError("Source token budget is too small")

    selected_text = select_document_text(document)

    reasoning_marker = "\n[SELECTED REASONING]\n"
    final_marker = "\n[FINAL SECTION]\n"

    if (
        reasoning_marker in selected_text
        and final_marker in selected_text
    ):
        lead, remainder = selected_text.split(
            reasoning_marker, 1
        )

        reasoning, final = remainder.split(
            final_marker, 1
        )

        reasoning_marker_ids = tokenizer.encode(
            reasoning_marker
        )
        final_marker_ids = tokenizer.encode(
            final_marker
        )

        content_budget = (
            budget
            - len(reasoning_marker_ids)
            - len(final_marker_ids)
        )

        if content_budget < 32:
            raise ValueError("Insufficient document budget")

        portions = [
            tokenizer.encode(lead),
            tokenizer.encode(reasoning),
            tokenizer.encode(final),
        ]

        # Allocate explicit space to each document section.
        quotas = [
            int(content_budget * 0.30),
            int(content_budget * 0.35),
        ]
        quotas.append(content_budget - sum(quotas))

        counts = [
            min(len(tokens), quota)
            for tokens, quota in zip(portions, quotas)
        ]

        # Redistribute unused space to other sections.
        remaining = content_budget - sum(counts)

        for index in (2, 1, 0):
            available = len(portions[index]) - counts[index]
            additional = min(remaining, available)

            counts[index] += additional
            remaining -= additional

        lead_ids = portions[0][:counts[0]]
        reasoning_ids = portions[1][:counts[1]]

        final_ids = (
            portions[2][-counts[2]:]
            if counts[2] > 0
            else []
        )

        document_ids = (
            lead_ids
            + reasoning_marker_ids
            + reasoning_ids
            + final_marker_ids
            + final_ids
        )

    else:
        document_ids = tokenizer.encode(selected_text)

        if len(document_ids) > budget:
            head_size = budget // 2
            tail_size = budget - head_size

            document_ids = (
                document_ids[:head_size]
                + document_ids[-tail_size:]
            )

    prompt_ids = (
        prefix_ids
        + document_ids
        + suffix_ids
    )

    assert len(prompt_ids) <= max_source_tokens

    return prompt_ids


class ConditionalSummarizationDataset(Dataset):
    def __init__(self, rows, tokenizer, max_source_tokens, max_target_tokens):
        self.rows = rows
        self.tokenizer = tokenizer
        self.source_tokens = max_source_tokens
        self.target_tokens = max_target_tokens
        self.end_ids = tokenizer.encode(END_SUMMARY)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        record = self.rows[index]
        prefix_ids = make_prompt_ids(
            record["document"], self.tokenizer, self.source_tokens
        )
        target_ids = self.tokenizer.encode(record["summary"])
        target_ids = target_ids[:self.target_tokens]
        if not target_ids:
            target_ids = self.tokenizer.encode("No summary available.")

        full = prefix_ids + target_ids + self.end_ids
        input_ids = full[:-1]
        labels = full[1:]
        target_position = len(prefix_ids) - 1
        labels[:target_position] = [-100] * target_position
        return torch.tensor(input_ids), torch.tensor(labels)


def collate_conditional(batch):
    max_len = max(len(tokens) for tokens, _ in batch)
    input_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros(len(batch), max_len, dtype=torch.bool)

    for row, (tokens, targets) in enumerate(batch):
        length = len(tokens)
        input_ids[row, :length] = tokens
        labels[row, :length] = targets
        attention_mask[row, :length] = True

    return input_ids, labels, attention_mask


In [23]:
"""Supervised and distillation training, conditional decoding and checkpoints."""

import gc
import json
import os
from pathlib import Path
import random
import time
import zipfile

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


def seed_everything(seed=781):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def canonical_device(device):
    """Turn a CUDA default-device alias into its concrete GPU index.

    In PyTorch `cuda` and `cuda:0` are not equal as Device objects, even
    though they refer to the same GPU when the active CUDA device is zero.
    """
    requested = torch.device(device)
    if requested.type == "cuda" and requested.index is None:
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested but no CUDA GPU is available")
        return torch.device("cuda", torch.cuda.current_device())
    return requested


def ensure_model_device(model, device, *, move=True):
    """Validate all parameters and buffers, not merely the model input.

    `move=True` repairs newly added CPU adapter weights before optimizer
    construction. A detailed error is raised only if movement fails.
    """
    expected = canonical_device(device)
    if move:
        model.to(expected)

    wrong = []
    for name, tensor in list(model.named_parameters()) + list(model.named_buffers()):
        if tensor.device != expected:
            wrong.append(f"{name}: {tensor.device} (expected {expected})")

    if wrong:
        raise RuntimeError(
            "Model has tensors on the wrong device after transfer:\n"
            + "\n".join(wrong[:12])
        )
    return model


def find_phase1_asset(asset_dir, filename):
    root = Path(asset_dir)
    matches = sorted(root.rglob(filename)) if root.is_dir() else []
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {filename} below {asset_dir}. "
            f"Found {len(matches)}. Set ASSET_DIR to matching model artifacts."
        )
    return matches[0]


def initialize_phase1(model_id, config, device, *, strict=True):
    tokenizer_path = find_phase1_asset(config["asset_dir"], "custom_bpe_scratch.json")
    phase1_path = find_phase1_asset(config["asset_dir"], "pytorch_model.bin")
    tokenizer = CustomByteLevelBPE().load(tokenizer_path)

    model = CustomLegalModel(
        model_id=model_id,
        vocab_size=config["vocab_size"],
        dim=config["hidden_size"],
        n_layers=config["n_layers"],
        n_heads=config["n_heads"],
        n_kv_heads=config["n_kv_heads"],
        max_seq_len=config["model_context"],
    )
    payload = torch.load(phase1_path, map_location="cpu", weights_only=True)
    if isinstance(payload, dict):
        for key in ("model_state_dict", "state_dict"):
            if key in payload:
                payload = payload[key]
                break
    model.load_state_dict(payload, strict=strict)
    if len(tokenizer.vocab) > config["vocab_size"]:
        raise ValueError("Tokenizer contains more IDs than model's embedding table")

    return ensure_model_device(model, device), tokenizer, phase1_path


def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    active = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return {"total": total, "trainable": active, "trainable_percent": 100 * active / total}


def batch_loss(model, input_ids, labels, attention_mask):
    logits, _ = model(input_ids, attention_mask=attention_mask)
    loss = F.cross_entropy(
        logits.float().reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100,
    )
    return loss, logits


def make_loader(rows, tokenizer, config, training):
    dataset = ConditionalSummarizationDataset(
        rows,
        tokenizer,
        max_source_tokens=config["max_source_tokens"],
        max_target_tokens=config["max_target_tokens"],
    )
    return DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=training,
        num_workers=0,
        collate_fn=collate_conditional,
    )


@torch.no_grad()
def average_validation_loss(model, loader, device, max_batches=None):
    model.eval()
    total = 0.0
    count = 0
    for batch_id, (tokens, targets, mask) in enumerate(loader):
        if max_batches is not None and batch_id >= max_batches:
            break
        tokens, targets, mask = (
            tokens.to(device), targets.to(device), mask.to(device)
        )
        loss, _ = batch_loss(model, tokens, targets, mask)
        total += float(loss.item())
        count += 1
    return total / max(1, count)


def save_adapter_checkpoint(model, path, metadata):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {"trainable_state_dict": trainable_parameter_dict(model), "metadata": metadata},
        path,
    )


def load_adapter_checkpoint(model, path, *, device):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    restore_trainable_parameters(model, payload["trainable_state_dict"])
    ensure_model_device(model, device)
    return payload.get("metadata", {})


def train_supervised(model, train_rows, val_rows, tokenizer, config, output_dir, device):
    device = canonical_device(device)
    ensure_model_device(model, device)
    train_loader = make_loader(train_rows, tokenizer, config, training=True)
    val_loader = make_loader(val_rows, tokenizer, config, training=False)
    parameters = [p for p in model.parameters() if p.requires_grad]
    assert parameters, "No trainable parameters"

    optimizer = torch.optim.AdamW(
        parameters,
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    grad_accumulation = config["gradient_accumulation"]
    amp_enabled = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    history = []
    best_loss = float("inf")
    checkpoint = Path(output_dir) / "adapter_best.pt"

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        counter = 0
        start = time.monotonic()

        for batch_id, (tokens, targets, mask) in enumerate(train_loader):
            tokens, targets, mask = (
                tokens.to(device), targets.to(device), mask.to(device)
            )
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16 if device.type == "cuda" else torch.bfloat16,
                enabled=amp_enabled,
            ):
                loss, _ = batch_loss(model, tokens, targets, mask)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite SFT loss: epoch={epoch}, batch={batch_id}")
            scaler.scale(loss / grad_accumulation).backward()
            is_last = batch_id == len(train_loader) - 1

            if (batch_id + 1) % grad_accumulation == 0 or is_last:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(parameters, config["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            running_loss += float(loss.detach())
            counter += 1
            if (batch_id + 1) % config["log_every"] == 0:
                print(
                    f"Epoch {epoch} batch {batch_id + 1}/{len(train_loader)} "
                    f"mean_loss={running_loss / counter:.4f}"
                )

        val_loss = average_validation_loss(
            model, val_loader, device, config["max_validation_batches"]
        )
        epoch_record = {
            "epoch": epoch,
            "training_loss": running_loss / max(counter, 1),
            "validation_loss": val_loss,
            "seconds": round(time.monotonic() - start, 2),
        }
        history.append(epoch_record)
        print(json.dumps(epoch_record, indent=2))

        if val_loss < best_loss:
            best_loss = val_loss
            save_adapter_checkpoint(
                model,
                checkpoint,
                {"epoch": epoch, "validation_loss": val_loss},
            )

    load_adapter_checkpoint(model, checkpoint, device=device)
    return history, checkpoint


def copy_merged_teacher_to_student(teacher, student):
    """Reduce depth, copying actual teacher blocks with compatible FFN topology."""
    teacher_depth = len(teacher.layers)
    student_depth = len(student.layers)

    if teacher.model_id == 4 and teacher_depth == 15 and student_depth == 8:
        indices = [0, 1, 4, 5, 8, 9, 12, 13]
    else:
        indices = [round(x * (teacher_depth - 1) / max(1, student_depth - 1))
                   for x in range(student_depth)]

    if student.model_id == 4:
        for i, selected in enumerate(indices):
            if i % 2 != selected % 2:
                raise ValueError("Student and teacher FFN parity mismatch")

    with torch.no_grad():
        teacher_embedding = (
            teacher.tok_embeddings if teacher.model_id == 2 else teacher.embeddings
        )
        student_embedding = (
            student.tok_embeddings if student.model_id == 2 else student.embeddings
        )
        student_embedding.weight.copy_(teacher_embedding.weight)
        student.norm.weight.copy_(teacher.norm.weight)

        for target_index, source_index in enumerate(indices):
            from_block = teacher.layers[source_index]
            to_block = student.layers[target_index]
            own_parameters = dict(to_block.named_parameters())
            source_parameters = dict(from_block.named_parameters())

            for name, target in own_parameters.items():
                if name in source_parameters:
                    target.copy_(source_parameters[name])
                elif name in {
                    "attn.wq.weight", "attn.wk.weight", "attn.wv.weight", "attn.wo.weight"
                }:
                    part = name.split(".")[1]
                    target.copy_(merge_trained_linear(getattr(from_block.attn, part)))
                else:
                    raise RuntimeError(f"Cannot distill teacher parameter {name}")
    return indices


def create_student(teacher, config, device):
    student = CustomLegalModel(
        model_id=teacher.model_id,
        vocab_size=config["vocab_size"],
        dim=teacher.dim,
        n_layers=config["student_layers"],
        n_heads=config["n_heads"],
        n_kv_heads=config["n_kv_heads"],
        max_seq_len=config["model_context"],
    )
    selected = copy_merged_teacher_to_student(teacher, student)
    return ensure_model_device(student, device), selected


def train_distilled_student(student, teacher, train_rows, val_rows, tokenizer,
                            config, output_dir, device):
    device = canonical_device(device)
    ensure_model_device(teacher, device)
    ensure_model_device(student, device)
    train_loader = make_loader(train_rows, tokenizer, config, training=True)
    val_loader = make_loader(val_rows, tokenizer, config, training=False)
    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=config["student_learning_rate"],
        weight_decay=config["weight_decay"],
    )
    amp_enabled = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    temperature = config["kd_temperature"]
    hard_weight = config["kd_hard_weight"]
    grad_accumulation = config["gradient_accumulation"]
    checkpoint = Path(output_dir) / "student_best.pt"
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    history = []
    best = float("inf")
    teacher.eval()

    for epoch in range(1, config["student_epochs"] + 1):
        student.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        count = 0
        start = time.monotonic()

        for batch_id, (tokens, targets, mask) in enumerate(train_loader):
            tokens, targets, mask = (
                tokens.to(device), targets.to(device), mask.to(device)
            )
            with torch.no_grad():
                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.float16 if amp_enabled else torch.bfloat16,
                    enabled=amp_enabled,
                ):
                    teacher_logits, _ = teacher(tokens, attention_mask=mask)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16 if amp_enabled else torch.bfloat16,
                enabled=amp_enabled,
            ):
                hard_loss, student_logits = batch_loss(student, tokens, targets, mask)

            valid = targets.reshape(-1) != -100
            flat_student = student_logits.float().reshape(-1, student_logits.size(-1))[valid]
            flat_teacher = teacher_logits.float().reshape(-1, teacher_logits.size(-1))[valid]

            softened_teacher = F.softmax(flat_teacher / temperature, dim=-1)
            softened_student = F.log_softmax(flat_student / temperature, dim=-1)
            soft_loss = F.kl_div(
                softened_student,
                softened_teacher,
                reduction="batchmean",
            ) * temperature ** 2
            loss = hard_weight * hard_loss + (1 - hard_weight) * soft_loss

            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite KD loss")
            scaler.scale(loss / grad_accumulation).backward()
            is_last = batch_id == len(train_loader) - 1
            if (batch_id + 1) % grad_accumulation == 0 or is_last:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    student.parameters(), config["grad_clip"]
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            running += float(loss.detach())
            count += 1
            if (batch_id + 1) % config["log_every"] == 0:
                print(f"KD epoch={epoch} batch={batch_id + 1} loss={running / count:.4f}")

        val_loss = average_validation_loss(
            student, val_loader, device, config["max_validation_batches"]
        )
        record = {
            "epoch": epoch,
            "kd_train_loss": running / max(count, 1),
            "student_validation_ce": val_loss,
            "seconds": round(time.monotonic() - start, 2),
        }
        history.append(record)
        print(json.dumps(record, indent=2))
        if val_loss < best:
            best = val_loss
            torch.save(student.state_dict(), checkpoint)

    student.load_state_dict(
        torch.load(checkpoint, map_location="cpu", weights_only=True), strict=True
    )
    return history, checkpoint


def _block_repeated_ngram(prefix, candidate, n=4):
    if len(prefix) < n - 1:
        return False
    context = tuple(prefix[-(n - 1):])
    past = {
        tuple(prefix[pos:pos + n])
        for pos in range(len(prefix) - n + 1)
    }
    return context + (candidate,) in past


@torch.inference_mode()
def generate_summary(model, tokenizer, document, config, device):
    device = canonical_device(device)
    ensure_model_device(model, device, move=False)
    model.eval()
    prompt = make_prompt_ids(document, tokenizer, config["max_source_tokens"])
    end_marker = tokenizer.encode(END_SUMMARY)
    generated = []
    cache = None
    next_input = torch.tensor([prompt], device=device, dtype=torch.long)
    start_position = 0

    for step in range(config["max_new_tokens"]):
        logits, cache = model(
            next_input,
            start_pos=start_position,
            kv_caches=cache,
        )
        scores = logits[0, -1].float()
        banned = generated
        if len(banned) >= 3:
            top_candidates = torch.topk(scores, min(60, scores.numel())).indices.tolist()
            for candidate in top_candidates:
                if _block_repeated_ngram(banned, candidate, n=4):
                    scores[candidate] = float("-inf")

        next_token = int(torch.argmax(scores))
        generated.append(next_token)
        if len(generated) >= len(end_marker) and generated[-len(end_marker):] == end_marker:
            generated = generated[:-len(end_marker)]
            break

        start_position += next_input.size(1)
        next_input = torch.tensor([[next_token]], device=device, dtype=torch.long)
        if start_position + 1 >= config["model_context"]:
            break

    summary = tokenizer.decode(generated).replace(END_SUMMARY, "").strip()
    return summary


def generate_predictions(model, rows, tokenizer, config, device, label):
    predictions = {}
    for index, record in enumerate(rows, start=1):
        summary = generate_summary(
            model, tokenizer, record["document"], config, device
        )
        if not summary:
            # A blank prediction is treated as a generation failure, not silently replaced.
            raise ValueError(f"Generated blank summary for {record['id']}")
        predictions[record["id"]] = summary
        if index % 25 == 0 or index == len(rows):
            print(f"{label}: {index}/{len(rows)} summaries generated")
    return predictions


def write_submission(test_rows, predictions, output_dir):
    official_ids = [row["id"] for row in test_rows]
    if set(predictions) != set(official_ids):
        raise ValueError("Mismatch with official test document IDs")
    if len(official_ids) != len(predictions):
        raise ValueError("Repeated official IDs")

    records = []
    for doc_id in official_ids:
        summary = predictions[doc_id]
        if not isinstance(summary, str) or not summary.strip():
            raise ValueError(f"Invalid summary for ID {doc_id}")
        records.append({"id": doc_id, "summary": summary.strip()})

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    json_path = out / "prediction.json"
    zip_path = out / "submission.zip"
    json_path.write_text(
        json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(json_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as archive:
        if archive.namelist() != ["prediction.json"]:
            raise AssertionError("Submission packaging error")
    return json_path, zip_path


def run_experiment(model_id, method, config, train_rows, val_rows, test_rows):
    seed_everything(config["seed"])
    device = canonical_device("cuda" if torch.cuda.is_available() else "cpu")
    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    model, tokenizer, phase1_path = initialize_phase1(
        model_id, config, device, strict=True
    )
    if config["max_source_tokens"] + config["max_target_tokens"] + 40 > config["model_context"]:
        raise ValueError("Training tokens exceed the Phase-1 model's context length")

    teacher_history = None
    teacher_info = {}
    if method == "kd":
        teacher = apply_adaptation(model, "lora", config)
        ensure_model_device(teacher, device)
        teacher_path = Path(config["teacher_checkpoint"])
        if teacher_path.is_file():
            teacher_info = load_adapter_checkpoint(
                teacher, teacher_path, device=device
            )
            print("Loaded previously fine-tuned LoRA teacher:", teacher_path)
        elif config["train_teacher_if_missing"]:
            print("No teacher checkpoint: training a task-specific LoRA teacher first")
            teacher_dir = output_dir / "teacher_training"
            teacher_history, teacher_path = train_supervised(
                teacher,
                train_rows,
                val_rows,
                tokenizer,
                config,
                teacher_dir,
                device,
            )
        else:
            raise FileNotFoundError(f"Missing fine-tuned teacher {teacher_path}")

        teacher.eval()
        student, selected_layers = create_student(teacher, config, device)
        print("Teacher parameters:", count_parameters(teacher))
        print("Student parameters:", count_parameters(student))
        history, best_checkpoint = train_distilled_student(
            student,
            teacher,
            train_rows,
            val_rows,
            tokenizer,
            config,
            output_dir,
            device,
        )
        model = student
        teacher_info.update({
            "teacher_path": str(teacher_path),
            "student_source_layers": selected_layers,
            "teacher_history": teacher_history,
        })
        del teacher
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        model = apply_adaptation(model, method, config)
        # New adapter weights must be on the same GPU as the pretrained model.
        ensure_model_device(model, device)
        print("Adaptation parameters:", count_parameters(model))
        history, best_checkpoint = train_supervised(
            model, train_rows, val_rows, tokenizer, config, output_dir, device
        )

    # Local lexical evaluation is descriptive only; primary BERTScore is separate.
    num_diagnostics = min(len(val_rows), config["validation_generation_count"])
    validation_rows = val_rows[:num_diagnostics]
    val_predictions = generate_predictions(
        model, validation_rows, tokenizer, config, device, label="Validation"
    )
    test_predictions = generate_predictions(
        model, test_rows, tokenizer, config, device, label="Test"
    )
    prediction_path, archive_path = write_submission(
        test_rows, test_predictions, output_dir
    )

    metadata = {
        "model_id": model_id,
        "method": method,
        "original_checkpoint": str(phase1_path),
        "best_training_checkpoint": str(best_checkpoint),
        "parameters": count_parameters(model),
        "history": history,
        "kd_info": teacher_info,
        "validation_predictions": val_predictions,
        "validation_references": {
            row["id"]: row["summary"] for row in validation_rows
        },
        "configuration": {
            key: value for key, value in config.items()
            if key not in ("hf_token",)
        },
        "primary_metric_note": "BERTScore not computed: use instructor's official scoring script",
    }
    record_path = output_dir / "run_record.json"
    record_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    return {
        "record_path": record_path,
        "prediction_path": prediction_path,
        "archive_path": archive_path,
        "metadata": metadata,
    }


## 5A · Device and adaptation preflight (runs before dataset download)

This tiny synthetic check catches CPU/GPU mismatches in newly inserted adaptation layers **before** expensive training. It does not train on synthetic samples or affect the real experiment's weights.


In [24]:
# A tiny forward/backward check on the device selected by the real experiment.
# This runs independently of the official training data and Phase-1 weights.
preflight_device = canonical_device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

preflight_config = {
    **CFG,
    "vocab_size": 256,
    "hidden_size": 32,
    "n_layers": 2,
    "n_heads": 4,
    "n_kv_heads": 2,
    "model_context": 128,
    "student_layers": 1,
}

probe = CustomLegalModel(
    model_id=MODEL_ID,
    vocab_size=256,
    dim=32,
    n_layers=2,
    n_heads=4,
    n_kv_heads=2,
    max_seq_len=128,
)

probe_method = "lora" if METHOD == "kd" else METHOD
probe = apply_adaptation(probe, probe_method, preflight_config)
ensure_model_device(probe, preflight_device)

probe_tokens = torch.randint(
    low=1,
    high=255,
    size=(1, 12),
    device=preflight_device,
)
probe_logits, _ = probe(probe_tokens)
probe_logits.float().mean().backward()

assert all(
    parameter.grad is None or torch.isfinite(parameter.grad).all()
    for parameter in probe.parameters()
)
assert any(
    parameter.grad is not None
    for parameter in probe.parameters()
    if parameter.requires_grad
)

if METHOD == "kd":
    student_probe, _ = create_student(
        probe, preflight_config, preflight_device
    )
    student_logits, _ = student_probe(probe_tokens)
    student_logits.float().mean().backward()
    assert any(
        parameter.grad is not None
        for parameter in student_probe.parameters()
        if parameter.requires_grad
    )
    del student_probe, student_logits

del probe, probe_logits, probe_tokens
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Device preflight PASS: {MODEL_ID=} {METHOD=} {preflight_device=}")


Device preflight PASS: MODEL_ID=4 METHOD='lorma' preflight_device=device(type='cuda', index=0)


## 6 · Official Phase 6 data

Only official training records are used for gradient updates. Validation is held out by **document ID**; test summaries have no public references.

In [25]:
all_rows = load_data("train", local_root=LOCAL_DATA_ROOT)
test_rows = load_data("test", local_root=LOCAL_DATA_ROOT)
train_rows, val_rows = split_train_validation(
    all_rows,
    fraction=0.10,
    seed=CFG["seed"],
)

print("Train documents:", len(train_rows))
print("Validation documents:", len(val_rows))
print("Test documents:", len(test_rows))
print("Sample training fields:", list(train_rows[0]))
print("Sample test fields:", list(test_rows[0]))
assert set(r["id"] for r in train_rows).isdisjoint(
    set(r["id"] for r in val_rows)
)

Train documents: 6327
Validation documents: 703
Test documents: 100
Sample training fields: ['id', 'document', 'summary']
Sample test fields: ['id', 'document']


## 7 · Train / distill / generate

The original model is loaded with **strict checkpoint matching**. For KD, a trained LoRA teacher is loaded when available; otherwise the notebook trains one before distilling. Validation loss is used for checkpoint selection.

In [26]:
result = run_experiment(
    model_id=MODEL_ID,
    method=METHOD,
    config=CFG,
    train_rows=train_rows,
    val_rows=val_rows,
    test_rows=test_rows,
)

print("Best checkpoint:", result["metadata"]["best_training_checkpoint"])
print("Prediction JSON:", result["prediction_path"])
print("Codabench ZIP:", result["archive_path"])

Adaptation parameters: {'total': 53620560, 'trainable': 403200, 'trainable_percent': 0.7519503712754958}
Epoch 1 batch 50/6327 mean_loss=3.0336
Epoch 1 batch 100/6327 mean_loss=3.0499
Epoch 1 batch 150/6327 mean_loss=3.0170
Epoch 1 batch 200/6327 mean_loss=3.0399
Epoch 1 batch 250/6327 mean_loss=3.0324
Epoch 1 batch 300/6327 mean_loss=3.0340
Epoch 1 batch 350/6327 mean_loss=3.0260
Epoch 1 batch 400/6327 mean_loss=3.0249
Epoch 1 batch 450/6327 mean_loss=3.0120
Epoch 1 batch 500/6327 mean_loss=3.0133
Epoch 1 batch 550/6327 mean_loss=3.0179
Epoch 1 batch 600/6327 mean_loss=3.0114
Epoch 1 batch 650/6327 mean_loss=3.0055
Epoch 1 batch 700/6327 mean_loss=3.0063
Epoch 1 batch 750/6327 mean_loss=3.0047
Epoch 1 batch 800/6327 mean_loss=2.9982
Epoch 1 batch 850/6327 mean_loss=2.9989
Epoch 1 batch 900/6327 mean_loss=2.9992
Epoch 1 batch 950/6327 mean_loss=2.9957
Epoch 1 batch 1000/6327 mean_loss=2.9924
Epoch 1 batch 1050/6327 mean_loss=2.9880
Epoch 1 batch 1100/6327 mean_loss=2.9865
Epoch 1 batch

In [27]:
import statistics

metadata = result["metadata"]

references = metadata["validation_references"]
predictions = metadata["validation_predictions"]

reference_lengths = [
    len(text.split())
    for text in references.values()
]

prediction_lengths = [
    len(text.split())
    for text in predictions.values()
]

print("Validation documents:", len(references))
print("Median reference words:", statistics.median(reference_lengths))
print("Median generated words:", statistics.median(prediction_lengths))
print("Maximum reference words:", max(reference_lengths))
print("Maximum generated words:", max(prediction_lengths))

for doc_id in list(references)[:2]:
    print("\nDocument:", doc_id)
    print("REFERENCE:", references[doc_id][:700])
    print("GENERATED:", predictions[doc_id][:700])

Validation documents: 30
Median reference words: 610.5
Median generated words: 576.0
Maximum reference words: 1683
Maximum generated words: 576

Document: 4058
REFERENCE: The appellant was detained under the in September, 1974 but was released in December, 1974. In February, 1977 he was detained under section 3(1) of the , on the ground that he was "likely to engage in transporting smuggled goods". One of the grounds of .detention stated: "you were an associate of a notorious smuggler, that you were engaged in piloting smuggled goods loaded in trucks from the place of landing . " In support of the above, two instances were mentioned: one relating to an incident on 6th August, 1974 and another on 25th August, 1974. In the appellant 's writ petition the High Court observed (1) that although some of the activities attributed to the detenu related to August 1974,
GENERATED: The appellant was a person who was the same and the appellant had been entitled to the benefit of the land in the Sta

In [28]:
tokenizer = CustomByteLevelBPE().load(
    find_phase1_asset(
        CFG["asset_dir"],
        "custom_bpe_scratch.json",
    )
)

row = next(
    row for row in val_rows
    if row["id"] == "4058"
)

prompt_ids = make_prompt_ids(
    row["document"],
    tokenizer,
    CFG["max_source_tokens"],
)

print("Original document words:", len(row["document"].split()))
print("Model input tokens:", len(prompt_ids))

print("\n=== ACTUAL INPUT SEEN BY MODEL ===")
print(tokenizer.decode(prompt_ids))

Original document words: 2787
Model input tokens: 768

=== ACTUAL INPUT SEEN BY MODEL ===
Summarize the following Indian legal judgment accurately. Cover the facts, the central legal issue, the court reasoning, and the outcome. Avoid introducing facts that are not in the judgment.

JUDGMENT:
N: Criminal Appeal No. 105 of 1978 with W.P. No. 833 of 1978. (Appeal by special leave from the Judgment and order dt. 27.9.77 of the Gujarat High Court in Criminal Spl. Application No. 176 of 1977) . Ram Jethmalani and H. section Parihar for the appellant in Cr. A. No. 105 and WP No. 833/78. M. N. Phadke and B. D. Sharma and M. N. Shroff for respondent No 1 Girish Chandra for respondent no 4. M. N. Shroff for respondents 1 3. R.B.Datar and Miss A. Subhashini for respondents 4 5. The
[SELECTED REASONING]
e in transporting smuggled goods in the areas around Balashwar and Sachin Gabheni Road in the State of Gujarat which are areas highly vulnerable to smuggling as defined in section 9 of the COFEPOSA

In [29]:
import statistics

tokenizer = CustomByteLevelBPE().load(
    find_phase1_asset(
        CFG["asset_dir"],
        "custom_bpe_scratch.json",
    )
)

sample = train_rows[:100]

lengths = [
    len(tokenizer.encode(row["summary"]))
    for row in sample
]

print("Sample size:", len(lengths))
print("Median reference tokens:", statistics.median(lengths))
print("Maximum reference tokens:", max(lengths))

print(
    "Percentage truncated during training:",
    round(
        100 * sum(
            length > CFG["max_target_tokens"]
            for length in lengths
        ) / len(lengths),
        2,
    ),
    "%",
)

Sample size: 100
Median reference tokens: 1550.5
Maximum reference tokens: 7272
Percentage truncated during training: 62.0 %


## 8 · Local ROUGE diagnostics

These are **descriptive** local ROUGE scores from generated validation summaries. Official BERTScore F1 is not calculated here and must be evaluated with the course scorer.

In [30]:
from collections import Counter
import re


def _words(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)


def rouge_1_f1(reference, prediction):
    ref_counts = Counter(_words(reference))
    pred_counts = Counter(_words(prediction))
    overlap = sum((ref_counts & pred_counts).values())
    precision = overlap / max(1, sum(pred_counts.values()))
    recall = overlap / max(1, sum(ref_counts.values()))
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def rouge_l_f1(reference, prediction):
    a, b = _words(reference), _words(prediction)
    row = [0] * (len(b) + 1)
    for token in a:
        previous = 0
        for j, other in enumerate(b, 1):
            old = row[j]
            row[j] = previous + 1 if token == other else max(row[j], row[j - 1])
            previous = old
    matches = row[-1]
    precision = matches / max(1, len(b))
    recall = matches / max(1, len(a))
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


references = result["metadata"]["validation_references"]
predictions = result["metadata"]["validation_predictions"]
rouge1 = [rouge_1_f1(references[k], v) for k, v in predictions.items()]
rougel = [rouge_l_f1(references[k], v) for k, v in predictions.items()]

print("Evaluated validation documents:", len(predictions))
print("ROUGE-1 F1 (local diagnostic):", sum(rouge1) / max(1, len(rouge1)))
print("ROUGE-L F1 (local diagnostic):", sum(rougel) / max(1, len(rougel)))
print("Primary BERTScore F1: not evaluated here")

if predictions:
    sample_id = next(iter(predictions))
    print("\nDocument ID:", sample_id)
    print("\nReference summary:\n", references[sample_id][:1200])
    print("\nGenerated summary:\n", predictions[sample_id][:1200])

Evaluated validation documents: 30
ROUGE-1 F1 (local diagnostic): 0.3805045504373953
ROUGE-L F1 (local diagnostic): 0.16908609093237462
Primary BERTScore F1: not evaluated here

Document ID: 4058

Reference summary:
 The appellant was detained under the in September, 1974 but was released in December, 1974. In February, 1977 he was detained under section 3(1) of the , on the ground that he was "likely to engage in transporting smuggled goods". One of the grounds of .detention stated: "you were an associate of a notorious smuggler, that you were engaged in piloting smuggled goods loaded in trucks from the place of landing . " In support of the above, two instances were mentioned: one relating to an incident on 6th August, 1974 and another on 25th August, 1974. In the appellant 's writ petition the High Court observed (1) that although some of the activities attributed to the detenu related to August 1974, the fact that the detenu was in illegal employment of a notorious smuggler under d

## 9 · Submission integrity

The generated ZIP contains a single root-level `prediction.json` with **every official test ID exactly once**. Inspect generation quality and the file before submitting to Codabench.

In [31]:
import zipfile

with zipfile.ZipFile(result["archive_path"]) as archive:
    names = archive.namelist()
    assert names == ["prediction.json"], names
    submitted = json.loads(archive.read("prediction.json"))

assert [item["id"] for item in submitted] == [row["id"] for row in test_rows]
assert all(isinstance(row["summary"], str) and row["summary"].strip() for row in submitted)

print("Submission entries:", len(submitted))
print("ZIP structure:", names)
print("Ready for manual submission review:", result["archive_path"])

Submission entries: 100
ZIP structure: ['prediction.json']
Ready for manual submission review: /kaggle/working/phase6_model2_lora_v2/submission.zip


## 10 · Experiment record for your report

`run_record.json` retains training losses, validation generations, model/parameter counts, checkpoint paths and the editable configuration. Analyze strengths and failures yourself, compare with the instructor's BERTScore script, and disclose AI code assistance according to your instructor's approval.